In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Isang Qiskit Function ng Q-CTRL Fire Opal
*Tingnan ang [API reference](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Pangkalahatang-ideya
Gamit ang Fire Opal Optimization Solver, malulutas mo ang mga utility-scale na optimization problem sa quantum hardware nang hindi kailangan ng malalim na kaalaman sa quantum computing. I-input lang ang mataas na antas na depinisyon ng problema, at ang Solver ang bahala sa lahat. Ang buong workflow ay may kamalayan sa ingay at gumagamit ng [Fire Opal's Performance Management](/guides/q-ctrl-performance-management) sa likod ng sistema. Patuloy na nagbibigay ang Solver ng tumpak na solusyon sa mga problemang mahirap lutasin nang klasikal, kahit sa pinakamataas na sukat ng pinakamalalaking IBM&reg; QPU.

Flexible ang Solver at magagamit para sa mga combinatorial optimization problem na tinukoy bilang objective function o arbitrary graph. Hindi kailangang i-map ang mga problema sa device topology. Malulutas ang parehong unconstrained at constrained na mga problema, basta ma-formulate ang mga constraint bilang penalty term. Ipinapakita ng mga halimbawa sa gabay na ito kung paano lutasin ang isang unconstrained at isang constrained na utility-scale optimization problem gamit ang iba't ibang uri ng Solver input. Ang unang halimbawa ay isang max-cut problem na tinukoy sa isang 156-node, 3-Regular na graph, habang ang pangalawa ay isang 50-node na Minimum Vertex Cover problem na tinukoy ng isang cost function.

Para makakuha ng access sa Optimization Solver, [makipag-ugnayan sa Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).

## Paglalarawan ng function
Ganap na ino-optimize at ino-automate ng Solver ang buong algorithm, mula sa error suppression sa hardware level hanggang sa mahusay na problem mapping at closed-loop classical optimization. Sa likod ng sistema, binabawasan ng pipeline ng Solver ang mga error sa bawat yugto, na nagbibigay ng mas mataas na performance na kailangan para makapag-scale nang maayos. Ang pinagbabatayang workflow ay inspirado ng Quantum Approximate Optimization Algorithm (QAOA), na isang hybrid quantum-classical na algorithm. Para sa detalyadong buod ng buong Optimization Solver workflow, tingnan ang [nai-publish na manuskrito](https://arxiv.org/abs/2406.01743).

![Visualization ng Optimization Solver workflow](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Para malutas ang isang pangkalahatang problema gamit ang Optimization Solver:
1. Tukuyin ang iyong problema bilang objective function, graph, o `SparsePauliOp` spin chain.
2. Kumonekta sa function sa pamamagitan ng Qiskit Functions Catalog.
3. Patakbuhin ang problema gamit ang Solver at kunin ang mga resulta.

### Tinatanggap na mga format ng problema
- Polynomial expression na representasyon ng isang objective function. Mas mainam na gawin ito sa Python gamit ang isang umiiral na SymPy Poly object at i-format sa string gamit ang [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Graph na representasyon ng isang partikular na uri ng problema. Dapat gawin ang graph gamit ang networkx library sa Python. Pagkatapos ay i-convert ito sa string gamit ang networkx function na `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Spin chain na representasyon ng isang partikular na problema. Dapat ireprensenta ang spin chain bilang `SparsePauliOp` object; tingnan ang [dokumentasyon](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) para sa karagdagang detalye.

> **Note:** Kung gusto mong gumamit ng backend na hindi pa sinusuportahan ng function na ito, [makipag-ugnayan sa Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) para idagdag ang suporta.
## Mga Benchmark
Ipinapakita ng [mga nai-publish na resulta ng benchmarking](https://arxiv.org/abs/2406.01743) na matagumpay na nireresulba ng Solver ang mga problema na may mahigit 120 qubit, at kahit pa nakakalampas sa mga dating nai-publish na resulta sa quantum annealing at trapped-ion na mga device. Ang mga sumusunod na benchmark metric ay nagbibigay ng magaspang na indikasyon ng katumpakan at scaling ng mga uri ng problema batay sa ilang halimbawa. Maaaring mag-iba ang aktwal na mga metric batay sa iba't ibang katangian ng problema, tulad ng bilang ng mga term sa objective function (density) at ang kanilang locality, bilang ng mga variable, at polynomial order.

Ang "Bilang ng mga qubit" na nakaindika ay hindi isang mahigpit na limitasyon kundi kumakatawan sa mga magaspang na threshold kung saan maaari kang umasa ng lubhang pare-pareho na katumpakan ng solusyon. Matagumpay na nareresulba ang mga mas malalaking sukat ng problema, at hinihikayat ang pagsubok na lagpas sa mga limitasyong ito.

Sinusuportahan ang arbitrary qubit connectivity sa lahat ng uri ng problema.

| Uri ng problema    | Bilang ng mga qubit | Halimbawa | Katumpakan | Kabuuang oras (s) | Paggamit ng runtime (s) | Bilang ng mga iteration
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected na mga quadratic problem  | 156 | 3-regular max-cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected na mga quadratic problem | 50 | Fully-connected max-cut| 100%      |  1758    | 268  | 12 |
| Constrained na problema na may penalty term | 50 | Weighted Minimum Vertex Cover na may 8% edge density | 100%      | 1074     | 215 | 10 |

## Magsimula
Una, mag-authenticate gamit ang iyong [IBM Quantum API key](http://quantum.cloud.ibm.com/). Pagkatapos, piliin ang Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na [na-save mo na ang iyong account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na kapaligiran.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Tukuyin ang problema
Maaari kang magpatakbo ng Max-Cut problem sa pamamagitan ng pagde-define ng graph problem at pagtukoy ng `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Patakbuhin ang problema
Kapag gumagamit ng graph-based na paraan ng input, tukuyin ang uri ng problema.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Kunin ang resulta
Kunin ang pinakamainam na halaga ng cut mula sa results dictionary.

> **Note:** Maaaring nagbago ang pagma-map ng mga variable sa bitstring. Ang output dictionary ay naglalaman ng `variables_to_bitstring_index_map` sub-dictionary, na tumutulong sa pag-verify ng pagkakasunod-sunod.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Maaari mong ma-verify ang katumpakan ng resulta sa pamamagitan ng paglutas ng problema nang klasikal gamit ang mga open-source solver tulad ng [PuLP](https://coin-or.github.io/pulp/) kung hindi masyadong siksik ang koneksyon ng graph. Para sa mga problemang may mataas na density, maaaring kailangan ng mas advanced na classical solver para ma-validate ang solusyon.

## Halimbawa: Constrained na optimization
Ang nakaraang max-cut example ay isang karaniwang quadratic unconstrained binary optimization problem. Maaaring gamitin ang Q-CTRL's Optimization Solver para sa iba't ibang uri ng problema, kasama na ang constrained optimization. Maaari kang lumutas ng arbitrary na uri ng problema sa pamamagitan ng pag-input ng depinisyon ng problema bilang polynomial kung saan ang mga constraint ay kinakatawan bilang penalty term.

Ipinapakita ng sumusunod na halimbawa kung paano bumuo ng cost function para sa isang constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Bukod sa mga package na `qiskit-ibm-catalog` at `qiskit`, gagamitin mo rin ang mga sumusunod na package para patakbuhin ang halimbawang ito: `numpy`, `networkx`, at `sympy`. Maaari mong i-install ang mga package na ito sa pamamagitan ng pag-uncomment ng sumusunod na cell kung pinapatakbo mo ang halimbawang ito sa isang notebook gamit ang IPython kernel.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Tukuyin ang problema
Tukuyin ang isang random na MVC problem sa pamamagitan ng paglikha ng graph na may mga node na may random na timbang.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Output ng nakaraang code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Ang isang karaniwang optimization model para sa weighted MVC ay maaaring i-formulate tulad ng sumusunod. Una, kailangan magdagdag ng penalty para sa anumang kaso kung saan ang isang edge ay hindi konektado sa isang vertex sa subset. Kaya, hayaan ang $n_i = 1$ kung ang vertex $i$ ay nasa cover (ibig sabihin, nasa subset) at $n_i = 0$ kung hindi. Pangalawa, ang layunin ay i-minimize ang kabuuang bilang ng mga vertex sa subset, na maaaring irepresenta ng sumusunod na function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Ngayon, ang bawat edge sa graph ay dapat may kasamang kahit isang endpoint mula sa cover, na maaaring ipahayag bilang inequality:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Ang anumang kaso kung saan ang isang edge ay hindi konektado sa vertex ng cover ay dapat parusahan. Maaari itong irepresenta sa cost function sa pamamagitan ng pagdaragdag ng penalty na may anyo $P(1-n_i-n_j+n_i n_j)$ kung saan ang $P$ ay isang positibong penalty constant. Kaya, ang isang unconstrained na alternatibo sa constrained inequality para sa weighted MVC ay:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Patakbuhin ang problema

In [17]:
print(mvc_job.status())

QUEUED


Suriin ang [status](/guides/functions#check-job-status) ng iyong Qiskit Function workload o kunin ang [mga resulta](/guides/functions#retrieve-results) tulad ng sumusunod:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>